# Day 7 lab, Part C: export and inspect

**Goal.** You export one PyTorch model to ONNX with a static shape, to ONNX
with a dynamic shape, and to LiteRT. You look at the graphs before and after
the optimization, and you find the fused operators.

**Deliverable.** The model files in the folder `models/` for the Raspberry
Pi, and the tables of Part C in the file `report.md`.

**Credits.** The graph optimizations (constant folding, operator fusion, the
folding of a batch normalization into a convolution) follow chapter 11
"Hardware Acceleration" and chapter 13 "Model Serving" of "Machine Learning
Systems" by Vijay Janapa Reddi and contributors (mlsysbook.ai,
CC BY-NC-SA 4.0). The model is MobileNetV2 of torchvision with its ImageNet
weights (BSD-3-Clause). All code is new code of this course. Keep this cell.

**Time.** 45 minutes. The complete notebook runs in about 2 minutes on the
work computer of the course.

**How to work.** Run the cells in order. A cell with the mark
`TODO (student)` has a task. The notebook has three tasks. The cell after a
task checks your result. A section that needs a task runs only when the task
is complete.

## 0. Setup

Run this cell first. It loads the packages and makes the folder `models/`.
The first run downloads the weights of the model: 14 MB.

The cell defines four helpers:

- `quiet()`: hides the many status lines that the exporters print.
- `onnx_nodes(path)`: the number of nodes of each operator type in an ONNX file.
- `litert_operators(path)`: the same for a LiteRT file.
- `median_ms(run)`: the median time of a function call in ms.

In [ ]:
import collections
import contextlib
import io
import logging
import os
import platform
import sys
import time
import warnings

warnings.filterwarnings("ignore")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
os.environ["LITERT_TORCH_SHOW_PROGRESS"] = "n"   # no progress lines of the LiteRT export


@contextlib.contextmanager
def quiet():
    '''Hide the status lines that an exporter prints.'''
    sys.stdout.flush()
    sys.stderr.flush()
    streams = [sys.stdout, sys.stderr]
    saved_write = [stream.write for stream in streams]
    saved_fd = os.dup(1), os.dup(2)
    logging.disable(logging.CRITICAL)
    with open(os.devnull, "w") as null:
        for stream in streams:
            stream.write = lambda *args, **kwargs: 0
        os.dup2(null.fileno(), 1)
        os.dup2(null.fileno(), 2)
        try:
            yield
        finally:
            os.dup2(saved_fd[0], 1)
            os.dup2(saved_fd[1], 2)
            os.close(saved_fd[0])
            os.close(saved_fd[1])
            for stream, write in zip(streams, saved_write):
                stream.write = write
            logging.disable(logging.NOTSET)


with quiet():
    import numpy as np
    import onnx
    import onnxruntime as ort
    import torch
    import torchvision
    import litert_torch
    from ai_edge_litert.interpreter import Interpreter

# The lab folder. The notebook of the folder solutions/ uses the folder above.
LAB = "." if os.path.isdir("models") else ".."
MODELS = os.path.join(LAB, "models")
os.makedirs(MODELS, exist_ok=True)

task_ok = {}


def onnx_nodes(path):
    '''Return a Counter: operator type -> number of nodes.'''
    graph = onnx.load(path).graph
    return collections.Counter(node.op_type for node in graph.node)


def litert_operators(path):
    '''Return a Counter: operator type -> number of operators.'''
    with quiet():
        interpreter = Interpreter(model_path=path)
        details = interpreter._get_ops_details()
    return collections.Counter(op["op_name"] for op in details)


def show(counter):
    '''Print a Counter in one line, the largest number first.'''
    parts = ["%s %d" % pair for pair in sorted(counter.items(), key=lambda p: (-p[1], p[0]))]
    print("  %d in total: %s" % (sum(counter.values()), ", ".join(parts)))


def median_ms(run, runs=30, warmup=5):
    '''Return the median time of run() in ms.'''
    for _ in range(warmup):
        run()
    times = []
    for _ in range(runs):
        start = time.perf_counter()
        run()
        times.append(1000 * (time.perf_counter() - start))
    return float(np.median(times))


def onnx_session(path, threads=1, level=None, save=None):
    '''Return an ONNX Runtime session for the CPU.'''
    options = ort.SessionOptions()
    options.intra_op_num_threads = threads
    options.log_severity_level = 3          # print errors only
    if level is not None:
        options.graph_optimization_level = level
    if save is not None:
        options.optimized_model_filepath = save
    return ort.InferenceSession(path, options, providers=["CPUExecutionProvider"])


print("Python", platform.python_version(), "on", platform.machine())
print("torch", torch.__version__, "| torchvision", torchvision.__version__)
print("onnx", onnx.__version__, "| onnxruntime", ort.__version__,
      "| litert-torch", litert_torch.__version__)
print("models folder:", os.path.normpath(MODELS))

## 1. The model (3 min)

The model is MobileNetV2 of torchvision with weights for the 1000 classes of
ImageNet. The cell counts its layers and makes one example input.

The mode `eval()` is necessary before each export. In this mode, a batch
normalization uses its stored statistics, and the dropout does nothing.

In [ ]:
model = torchvision.models.mobilenet_v2(weights="IMAGENET1K_V1").eval()

layers = collections.Counter(type(m).__name__ for m in model.modules()
                             if not list(m.children()))
print("Layers of the PyTorch model:")
show(layers)
print("Parameters:", sum(p.numel() for p in model.parameters()))

torch.manual_seed(0)
example = torch.randn(1, 3, 224, 224)
with torch.inference_mode():
    reference = model(example).numpy()
print("Example input:", tuple(example.shape), "| output:", reference.shape,
      "| largest output: class", int(reference.argmax()))

## 2. Task 1: export to ONNX with a static shape (7 min)

Write the function `export_static`. It calls `torch.onnx.export` with:

- the model, a tuple with the example input, and the path of the file
- `input_names=["image"]` and `output_names=["logits"]`
- `dynamo=True` and `external_data=False`

The option `external_data=False` puts the weights into the ONNX file. With no
such option, the exporter writes a second file for the weights.

In [ ]:
def export_static(model, example, path):
    '''Task 1. Write the model as an ONNX file with a static input shape.'''
    # TODO (student): call torch.onnx.export with the arguments of the text
    # above. The function returns nothing.
    return None

In [ ]:
STATIC = os.path.join(MODELS, "mnv2_static.onnx")
if os.path.exists(STATIC):
    os.remove(STATIC)
with quiet():
    export_static(model, example, STATIC)

task_ok[1] = False
if os.path.exists(STATIC):
    graph = onnx.load(STATIC).graph
    dims = [d.dim_value for d in graph.input[0].type.tensor_type.shape.dim]
    names = (graph.input[0].name, graph.output[0].name)
    output = onnx_session(STATIC).run(None, {graph.input[0].name: example.numpy()})[0]
    difference = float(np.abs(output - reference).max() / np.abs(reference).max())
    task_ok[1] = dims == [1, 3, 224, 224] and names == ("image", "logits") and difference < 1e-3
    print("Input", names[0], dims, "| output", names[1], "| file:",
          os.path.getsize(STATIC), "bytes")
    print("Largest difference to the PyTorch output: %.1e of the largest value" % difference)
    print("Nodes of the ONNX file:")
    show(onnx_nodes(STATIC))
print("Task 1:", "complete" if task_ok[1] else "not complete")

Compare the nodes of the file with the layers of section 1. Write the two
lists in the report. Which layer type of the PyTorch model has no node in the
file?

## 3. Task 2: export with a dynamic batch size (8 min)

Write the function `export_dynamic`. It is the same call with one more
argument:

```python
dynamic_shapes={"x": {0: torch.export.Dim("batch", min=1, max=64)}}
```

`"x"` is the name of the input in the function `forward` of the model.
Dimension 0 of this input gets the name `batch`.

In [ ]:
def export_dynamic(model, example, path):
    '''Task 2. Write the model as an ONNX file with a dynamic batch size.'''
    # TODO (student): the call of Task 1 with the argument dynamic_shapes.
    return None

In [ ]:
DYNAMIC = os.path.join(MODELS, "mnv2_dynamic.onnx")
if os.path.exists(DYNAMIC):
    os.remove(DYNAMIC)
with quiet():
    export_dynamic(model, example, DYNAMIC)

task_ok[2] = False
if os.path.exists(DYNAMIC):
    graph = onnx.load(DYNAMIC).graph
    dims = [d.dim_param or d.dim_value for d in graph.input[0].type.tensor_type.shape.dim]
    torch.manual_seed(1)
    three = torch.randn(3, 3, 224, 224)
    with torch.inference_mode():
        expected = model(three).numpy()
    try:
        output = onnx_session(DYNAMIC).run(None, {graph.input[0].name: three.numpy()})[0]
        good = output.shape == (3, 1000) and \
            float(np.abs(output - expected).max() / np.abs(expected).max()) < 1e-3
    except Exception as error:
        good = False
        print("The file does not accept 3 images:", str(error).split("\n")[0][:120])
    task_ok[2] = isinstance(dims[0], str) and dims[1:] == [3, 224, 224] and good
    print("Input shape of the file:", dims)
print("Task 2:", "complete" if task_ok[2] else "not complete")

The next cell compares the two files in ONNX Runtime with one thread. Before
you run it, write your prediction in the report:

- Is the dynamic file slower than the static file for one image?
- What does the static file do with 2 images?
- Is a batch of 4 images faster for each image than 4 runs with one image?

In [ ]:
if task_ok[1] and task_ok[2]:
    one = example.numpy()
    static_session = onnx_session(STATIC)
    dynamic_session = onnx_session(DYNAMIC)
    print("1 image, static file:  %5.1f ms" % median_ms(lambda: static_session.run(None, {"image": one})))
    print("1 image, dynamic file: %5.1f ms" % median_ms(lambda: dynamic_session.run(None, {"image": one})))
    try:
        static_session.run(None, {"image": np.concatenate([one, one])})
        print("2 images, static file: no error")
    except Exception as error:
        text = " ".join(str(error).split())
        print("2 images, static file: error.",
              text[text.find("Got invalid"):].split(" Please")[0])
    four = np.concatenate([one] * 4)
    start = time.perf_counter()
    dynamic_session.run(None, {"image": four})
    first = 1000 * (time.perf_counter() - start)
    later = median_ms(lambda: dynamic_session.run(None, {"image": four}), runs=10, warmup=2)
    print("4 images, dynamic file: first run %.1f ms, next runs %.1f ms, %.1f ms for each image"
          % (first, later, later / 4))
else:
    print("Tasks 1 and 2 are not complete. This section does not run.")

## 4. Export to LiteRT (5 min)

The package `litert-torch` converts a PyTorch model to a `.tflite` file. You
do not change this code. The cell prints the operators of the file.

Look at the list. The PyTorch model has 35 activations of the type ReLU6.
How many activation operators does the LiteRT file have? Write the answer in
the report.

In [ ]:
LITERT = os.path.join(MODELS, "mnv2.tflite")
with quiet():
    litert_torch.convert(model, (example,)).export(LITERT)

with quiet():
    interpreter = Interpreter(model_path=LITERT, num_threads=1)
    interpreter.allocate_tensors()
input_details = interpreter.get_input_details()[0]
output_details = interpreter.get_output_details()[0]
interpreter.set_tensor(input_details["index"], example.numpy())
interpreter.invoke()
litert_output = interpreter.get_tensor(output_details["index"])

print("File:", os.path.getsize(LITERT), "bytes | input",
      [int(v) for v in input_details["shape"]], input_details["dtype"].__name__)
print("Largest difference to the PyTorch output: %.1e of the largest value"
      % float(np.abs(litert_output - reference).max() / np.abs(reference).max()))
print("Operators of the LiteRT file:")
show(litert_operators(LITERT))

## 5. The graph before and after the optimization (12 min)

The exporter of section 2 already did a part of the optimization. To see each
step, the next cell makes one more file, `mnv2_unfused.onnx`. This file keeps
the batch normalizations of the model.

ONNX Runtime then loads this file four times, each time with a higher
optimization level. The option `optimized_model_filepath` writes the graph
that the runtime runs:

| File in `models/` | Optimization level |
|---|---|
| `mnv2_level_disabled.onnx` | none |
| `mnv2_level_basic.onnx` | basic |
| `mnv2_level_extended.onnx` | extended |
| `mnv2_level_all.onnx` | all |

Before you run the cell, write in the report: which level removes the batch
normalizations, and which level gives the largest gain in time?

In [ ]:
UNFUSED = os.path.join(MODELS, "mnv2_unfused.onnx")
with quiet():
    torch.onnx.export(model, (example,), UNFUSED,
                      input_names=["image"], output_names=["logits"],
                      dynamo=False, training=torch.onnx.TrainingMode.PRESERVE,
                      do_constant_folding=False)
model.eval()

levels = [("disabled", ort.GraphOptimizationLevel.ORT_DISABLE_ALL),
          ("basic", ort.GraphOptimizationLevel.ORT_ENABLE_BASIC),
          ("extended", ort.GraphOptimizationLevel.ORT_ENABLE_EXTENDED),
          ("all", ort.GraphOptimizationLevel.ORT_ENABLE_ALL)]
one = example.numpy()
print("%-9s %6s %9s %11s   %s" % ("level", "nodes", "load s", "latency ms", "node types"))
for name, level in levels:
    saved = os.path.join(MODELS, "mnv2_level_%s.onnx" % name)
    start = time.perf_counter()
    session = onnx_session(UNFUSED, level=level, save=saved)
    load = time.perf_counter() - start
    latency = median_ms(lambda: session.run(None, {"image": one}))
    count = onnx_nodes(saved)
    types = ", ".join("%s %d" % pair for pair in sorted(count.items(), key=lambda p: -p[1])[:4])
    print("%-9s %6d %9.3f %11.1f   %s" % (name, sum(count.values()), load, latency, types))

Now open these files in Netron (`netron.app`, or the command `netron` with a
file name): `mnv2_unfused.onnx`, `mnv2_level_basic.onnx`,
`mnv2_level_extended.onnx`, `mnv2_level_all.onnx`, and `mnv2.tflite`.

Answer the questions of Part C in the report:

1. In `mnv2_unfused.onnx`, which three nodes come after the input?
2. In `mnv2_level_basic.onnx`, which of these nodes is gone? Click the first
   `Conv` node: which new input does it have?
3. In `mnv2_level_extended.onnx`, what is the name of the first node? Which
   attribute holds the activation?
4. In `mnv2.tflite`, click the first `Conv2D` operator. Which property holds
   the activation? Which operator comes before it, and why?
5. The latency on your laptop: which level gives the largest gain? This gain
   comes from a memory layout that ONNX Runtime uses on an x86 processor. On
   the Raspberry Pi, the result can be different. You measure it in Part D.

## 6. Task 3: fold a batch normalization by hand (7 min)

The basic level folded each batch normalization into its convolution. Do the
same for the first layer of the model, with the formulas of the lecture:

$$s = \frac{\gamma}{\sqrt{\sigma^2 + \varepsilon}} \qquad W' = s\,W \qquad b' = \beta - s\,\mu$$

The convolution of this layer has no bias, so $b = 0$. Each output channel has
its own $s$. The weight tensor has the shape (output channels, input channels,
height, width). Multiply each filter by the $s$ of its channel.

Write the function `fold_batch_norm`. It returns the tensor $W'$ and the
tensor $b'$.

In [ ]:
def fold_batch_norm(weight, gamma, beta, mean, var, eps):
    '''Task 3. Return the weight and the bias of the fused convolution.'''
    # TODO (student): calculate s, then the new weight and the new bias.
    # s has one value for each output channel. Use s[:, None, None, None] to
    # multiply each filter by its value.
    return None

In [ ]:
conv, bn = model.features[0][0], model.features[0][1]
result = fold_batch_norm(conv.weight.detach(), bn.weight.detach(), bn.bias.detach(),
                         bn.running_mean, bn.running_var, bn.eps)

task_ok[3] = False
if result is not None:
    new_weight, new_bias = result
    with torch.inference_mode():
        two_operators = bn(conv(example))
        one_operator = torch.nn.functional.conv2d(example, new_weight, new_bias,
                                                  stride=conv.stride, padding=conv.padding)
    difference = float((two_operators - one_operator).abs().max())
    task_ok[3] = tuple(new_weight.shape) == tuple(conv.weight.shape) and difference < 1e-4
    print("Weight", tuple(new_weight.shape), "| bias", tuple(new_bias.shape))
    print("Largest difference between the two outputs: %.1e (largest output: %.2f)"
          % (difference, float(two_operators.abs().max())))
    channels = sum(m.num_features for m in model.modules()
                   if isinstance(m, torch.nn.BatchNorm2d))
    print("All batch normalizations of the model: %d channels, %d values. "
          "After the folding: %d bias values." % (channels, 4 * channels, channels))
print("Task 3:", "complete" if task_ok[3] else "not complete")

## 7. The same result in each runtime (3 min)

The cell gives the example input to each file and compares the output with
the output of PyTorch. The file `mnv2_int8.tflite` is in the lab folder. It is
the same model in `int8`, with a calibration on 100 images of ImageNet
classes.

One input is not a test of the accuracy. It shows only that the conversion
did not break the model.

In [ ]:
names = [line.strip() for line in open(os.path.join(MODELS, "imagenet_classes.txt"),
                                       encoding="utf-8")]


def report_output(name, output):
    difference = float(np.abs(output - reference).max() / np.abs(reference).max())
    print("%-22s class %3d %-18s difference %.1e"
          % (name, int(output.argmax()), names[int(output.argmax())], difference))


def run_litert(path, data):
    with quiet():
        interpreter = Interpreter(model_path=path, num_threads=1)
        interpreter.allocate_tensors()
    i = interpreter.get_input_details()[0]
    o = interpreter.get_output_details()[0]
    if i["dtype"] != np.float32:
        scale, zero_point = i["quantization"]
        data = np.clip(np.round(data / scale + zero_point), -128, 127).astype(i["dtype"])
    interpreter.set_tensor(i["index"], data)
    interpreter.invoke()
    output = interpreter.get_tensor(o["index"]).astype(np.float32)
    if o["dtype"] != np.float32:
        scale, zero_point = o["quantization"]
        output = (output - zero_point) * scale
    return output


report_output("PyTorch", reference)
if task_ok[1]:
    report_output("ONNX Runtime, float32", onnx_session(STATIC).run(None, {"image": example.numpy()})[0])
report_output("LiteRT, float32", run_litert(LITERT, example.numpy()))
report_output("LiteRT, int8", run_litert(os.path.join(MODELS, "mnv2_int8.tflite"), example.numpy()))

## 8. The files for the Raspberry Pi (2 min)

The cell lists the model files. Copy four of them to the Raspberry Pi for
Part D. Replace `NN` with the number of your group. Run the command in a
terminal, in the folder `Labs/day07/`:

```bash
scp models/mnv2.tflite models/mnv2_static.onnx models/mnv2_dynamic.onnx models/mnv2_unfused.onnx edge@pi-NN.local:~/edgeai/day07/models/
```

Do not copy the files `mnv2_level_*.onnx`. ONNX Runtime made them for the
processor of your laptop.

In [ ]:
print("%-28s %12s" % ("file", "bytes"))
for name in sorted(os.listdir(MODELS)):
    if name.endswith((".onnx", ".tflite")):
        print("%-28s %12d" % (name, os.path.getsize(os.path.join(MODELS, name))))
print()
print("Tasks complete:", ", ".join(str(n) for n in sorted(task_ok) if task_ok[n]) or "none")

## Summary

- An export writes the graph, the operators, and the weights into one file.
  The mode `eval()` comes first.
- A static shape is the default. A dynamic dimension is a decision at the
  export.
- The optimizer folds constants, folds each batch normalization into its
  convolution, and fuses each activation into its convolution.
- Each runtime shows the result in a different way: a `FusedConv` node in
  ONNX Runtime, and a property of the operator in LiteRT.
- The latency of your laptop is not the latency of the Raspberry Pi. Part D
  measures the files on the board.